# Module 4: Attention — Intuition & Small Implementation

## Learning Objectives
- Understand scaled dot-product attention mechanism
- Implement attention from scratch with NumPy/PyTorch
- Visualize attention maps
- Understand causal masking and Rotary Embedding
- Connect to Kyro's `LlamaAttention` and `PagedAttention`

## Concept: Scaled Dot-Product Attention

Attention computes a weighted sum of values, where weights are determined by query-key similarity:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q (Query)**: What we're looking for
- **K (Key)**: What's available
- **V (Value)**: The content associated with each key
- **Scaling**: $\frac{1}{\sqrt{d_k}}$ prevents large dot products

**Kyro mapping**: `src/model/llama.rs::LlamaAttention::forward` with `PagedAttention` for KV cache.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def np_softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask * -1e9
    attention_weights = np_softmax(scores, axis=-1)
    output = attention_weights @ V
    return output, attention_weights

# Create tiny example
seq_len = 4
d_k = 8
num_heads = 2

np.random.seed(42)
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, attn = scaled_dot_product_attention(Q, K, V)
print(f'Attention output shape: {output.shape}')
print(f'Attention weights shape: {attn.shape}')
print(f'Attention weights sum per row: {attn.sum(axis=-1)}')

## Step 1: Causal Masking

In [ ]:
# Create causal mask (lower triangular)
def causal_mask(seq_len):
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    return mask

mask = causal_mask(seq_len)
print('Causal mask:')
print(mask)

# Apply causal attention
output_causal, attn_causal = scaled_dot_product_attention(Q, K, V, mask=mask)
print('\nCausal attention weights:')
print(attn_causal)

## Step 2: Visualize Attention Maps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Unmasked attention
im1 = axes[0].imshow(attn, cmap='Blues', aspect='auto')
axes[0].set_title('Unmasked Attention')
axes[0].set_xlabel('Key positions')
axes[0].set_ylabel('Query positions')
plt.colorbar(im1, ax=axes[0])

# Causal attention
im2 = axes[1].imshow(attn_causal, cmap='Blues', aspect='auto')
axes[1].set_title('Causal Attention')
axes[1].set_xlabel('Key positions')
axes[1].set_ylabel('Query positions')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig('attention_maps.png', dpi=100)
plt.show()
print('Saved attention_maps.png')

## Step 3: Multi-Head Attention

In [ ]:
def multi_head_attention(Q, K, V, num_heads):
    seq_len, d_model = Q.shape
    d_k = d_model // num_heads
    
    # Split into heads
    Q_h = Q.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    K_h = K.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    V_h = V.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)

    outputs = []
    for i in range(num_heads):
        head_out, _ = scaled_dot_product_attention(Q_h[i], K_h[i], V_h[i])
        outputs.append(head_out)

    # Concatenate heads
    concat = np.concatenate(outputs, axis=-1).reshape(seq_len, d_model)
    return concat

output_mha = multi_head_attention(Q, K, V, num_heads=4)
print(f'Multi-head attention output shape: {output_mha.shape}')

## Step 4: Rotary Embedding Concept

In [ ]:
def rotary_embedding(dim, max_seq_len, device='cpu'):
    """Create rotary embedding frequencies."""
    inv_freq = np.array([
        1.0 / (10000.0 ** (2 * i / dim)) for i in range(dim // 2)
    ])
    return inv_freq

inv_freq = rotary_embedding(8, 2048)
print(f'Rotary embedding frequencies: {inv_freq}')
print('\nRotary embeddings rotate query/key vectors based on position,
      encoding relative position information without adding position tokens.')

print('\nIn Kyro: src/model/layers.rs::RotaryEmbedding::new()')
print('applies rotary embeddings in LlamaAttention::forward()')

## Exercises
1. Implement masked attention with different mask patterns.
2. Compare different head dimensions and their effect on attention.
3. Explore Kyro's `src/model/llama.rs::LlamaAttention` and `src/model/attention_kernel.rs::PagedAttention`.